In [1]:
import os
import csv

folder = r'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\EOIR Case Data'

In [ ]:
for filename in os.listdir(folder):

    if filename.endswith('.csv') and filename != 'tbl_schedule.csv':
        filepath = os.path.join(folder, filename)
        clean_name = filename.replace('.csv', '_clean.csv')
        clean_path = os.path.join(folder, clean_name)
        
        print(f'Cleaning {filename}...')
        
        with open(filepath, 'rb') as f:
            data = f.read().replace(b'\x00', b'')
        
        
        with open(clean_path, 'wb') as f:
            f.write(data)

print('cleaned!')

Cleaning A_TblCase.csv...
Cleaning A_TblCaseIdentifier.csv...
Cleaning B_TblProceedCharges.csv...
Cleaning B_TblProceeding.csv...
Cleaning D_TblAssociatedBond.csv...
Cleaning tblAppeal.csv...
Cleaning tblAppeal2.csv...
Cleaning tblAppealFedCourts.csv...
Cleaning tblProBono.csv...
Cleaning tblThreeMbrReferrals.csv...
Cleaning tbl_CasePriorityHistory.csv...
Cleaning tbl_Court_Appln.csv...
Cleaning tbl_Court_Motions.csv...
Cleaning tbl_CustodyHistory.csv...
Cleaning tbl_EOIR_Attorney.csv...
Cleaning tbl_JuvenileHistory.csv...
Cleaning tbl_Lead_Rider.csv...
Cleaning tbl_RepsAssigned.csv...
cleaned!


In [ ]:
with open(folder + r'\A_TblCase_clean.csv', 'rb') as f:
    first_line = f.readline().decode('utf-8')

cols = first_line.strip().split('\t')
print(f'Total columns: {len(cols)}')

for i, col in enumerate(cols):
    print(f'{i+1}: {col}')

Total columns: 39
1: IDNCASE
2: ALIEN_CITY
3: ALIEN_STATE
4: ALIEN_ZIPCODE
5: UPDATED_ZIPCODE
6: UPDATED_CITY
7: NAT
8: LANG
9: CUSTODY
10: SITE_TYPE
11: E_28_DATE
12: ATTY_NBR
13: CASE_TYPE
14: UPDATE_SITE
15: LATEST_HEARING
16: LATEST_TIME
17: LATEST_CAL_TYPE
18: UP_BOND_DATE
19: UP_BOND_RSN
20: CORRECTIONAL_FAC
21: RELEASE_MONTH
22: RELEASE_YEAR
23: INMATE_HOUSING
24: DATE_OF_ENTRY
25: C_ASY_TYPE
26: C_BIRTHDATE
27: C_RELEASE_DATE
28: UPDATED_STATE
29: ADDRESS_CHANGEDON
30: ZBOND_MRG_FLAG
31: Sex
32: DATE_DETAINED
33: DATE_RELEASED
34: LPR
35: DETENTION_DATE
36: DETENTION_LOCATION
37: DCO_LOCATION
38: DETENTION_FACILITY_TYPE
39: CASEPRIORITY_CODE


In [ ]:
with open(folder + r'\A_TblCase_clean.csv', 'rb') as f:

    for i, line in enumerate(f, 1):

        if i == 111549: # Postgres error line
            decoded = line.decode('utf-8')
            cols = decoded.strip().split('\t')
            print(f'Column count: {len(cols)}')

            for j, col in enumerate(cols):
                print(f'{j+1}: {repr(col)}')
                
            break

Column count: 40
1: '7375924'
2: 'BROOKLYN'
3: ''
4: 'NY'
5: '11209'
6: ''
7: ''
8: 'EG'
9: 'AR'
10: 'N'
11: 'M'
12: '2024-09-12 00:00:00.000'
13: '0'
14: 'RMV'
15: 'NYC'
16: '2014-12-02 00:00:00.000'
17: '1000'
18: 'M'
19: ''
20: ''
21: ''
22: ''
23: ''
24: ''
25: '1998-12-06 00:00:00.000'
26: 'I'
27: '4/1974'
28: ''
29: ''
30: ''
31: ''
32: 'F'
33: ''
34: ''
35: '0'
36: ''
37: ''
38: ''
39: ''
40: 'N/A'


In [ ]:
# Trim and pad columns so it is consistent throughout

input_file = folder + r'\A_TblCase_clean.csv'
output_file = folder + r'\A_TblCase_final.csv'

num_cols = 39

with open(input_file, 'rb') as infile, open(output_file, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile, delimiter='\t')

    for i, raw_line in enumerate(infile):
        line = raw_line.decode('utf-8', errors='replace').strip()
        cols = line.split('\t')

        if len(cols) > num_cols:
            cols = cols[:num_cols]  # trim extra columns

        elif len(cols) < num_cols:
            cols += [''] * (num_cols - len(cols))  # pad missing columns

        writer.writerow(cols)

print('done.')

done.


In [9]:
skip = ['tbl_schedule.csv', 'A_TblCase.csv', 'A_TblCase_clean.csv', 'A_TblCase_final.csv']
cleaned_list = []

for filename in os.listdir(folder):

    if filename.endswith('.csv') and filename not in skip and '_final' not in filename:
        filepath = os.path.join(folder, filename)
        final_path = os.path.join(folder, filename.replace('.csv', '_final.csv'))

        print(f'Currently processing: {filename}')

        # strip null bytes
        with open(filepath, 'rb') as f:
            data = f.read().replace(b'\x00', b'')

        lines = data.decode('utf-8', errors='replace').splitlines()

        # detect column count from header
        header = lines[0].strip().split('\t')
        num_cols = len(header)
        print(f'  Columns: {num_cols}')

        # normalize all rows
        with open(final_path, 'w', newline='', encoding='utf-8') as outfile:
            writer = csv.writer(outfile, delimiter='\t')

            for line in lines:
                cols = line.strip().split('\t')

                if len(cols) > num_cols:
                    cols = cols[:num_cols]

                elif len(cols) < num_cols:
                    cols += [''] * (num_cols - len(cols))
                    
                writer.writerow(cols)

        cleaned_file = filename.replace(".csv", "_final.csv")
        cleaned_list.append(cleaned_file)

        print(f'{filename.replace(".csv", "_final.csv")} --- complete.')

print(cleaned_list)

Currently processing: A_TblCaseIdentifier.csv
  Columns: 3
A_TblCaseIdentifier_final.csv --- complete.
Currently processing: B_TblProceedCharges.csv
  Columns: 5
B_TblProceedCharges_final.csv --- complete.
Currently processing: B_TblProceeding.csv
  Columns: 38
B_TblProceeding_final.csv --- complete.
Currently processing: D_TblAssociatedBond.csv
  Columns: 31
D_TblAssociatedBond_final.csv --- complete.
Currently processing: tblAppeal.csv
  Columns: 17
tblAppeal_final.csv --- complete.
Currently processing: tblAppeal2.csv
  Columns: 3
tblAppeal2_final.csv --- complete.
Currently processing: tblAppealFedCourts.csv
  Columns: 4
tblAppealFedCourts_final.csv --- complete.
Currently processing: tblProBono.csv
  Columns: 65
tblProBono_final.csv --- complete.
Currently processing: tblThreeMbrReferrals.csv
  Columns: 4
tblThreeMbrReferrals_final.csv --- complete.
Currently processing: tbl_CasePriorityHistory.csv
  Columns: 5
tbl_CasePriorityHistory_final.csv --- complete.
Currently processing: 

In [11]:
files = ['A_TblCaseIdentifier_final.csv', 'B_TblProceedCharges_final.csv', 'B_TblProceeding_final.csv', 'D_TblAssociatedBond_final.csv', 
         'tblAppeal_final.csv', 'tblAppeal2_final.csv', 'tblAppealFedCourts_final.csv', 'tblProBono_final.csv', 'tblThreeMbrReferrals_final.csv', 
         'tbl_CasePriorityHistory_final.csv', 'tbl_Court_Appln_final.csv', 'tbl_Court_Motions_final.csv', 'tbl_CustodyHistory_final.csv', 
         'tbl_EOIR_Attorney_final.csv', 'tbl_JuvenileHistory_final.csv', 'tbl_Lead_Rider_final.csv', 'tbl_RepsAssigned_final.csv']

for filename in files:
    filepath = os.path.join(folder, filename)

    with open(filepath, 'rb') as f:
        first_line = f.readline().decode('utf-8')

    columns = first_line.strip().split('\t')

    print(f'\n{filename}: {len(columns)}')

    for i, col in enumerate(columns):
        print(f'\t{col} varchar,')


A_TblCaseIdentifier_final.csv: 3
	IDNCASEID varchar,
	IDNCASE varchar,
	CASE_ID varchar,

B_TblProceedCharges_final.csv: 5
	IDNPRCDCHG varchar,
	IDNCASE varchar,
	IDNPROCEEDING varchar,
	CHARGE varchar,
	CHG_STATUS varchar,

B_TblProceeding_final.csv: 38
	IDNPROCEEDING varchar,
	IDNCASE varchar,
	OSC_DATE varchar,
	INPUT_DATE varchar,
	BASE_CITY_CODE varchar,
	HEARING_LOC_CODE varchar,
	IJ_CODE varchar,
	TRANS_IN_DATE varchar,
	PREV_HEARING_LOC varchar,
	PREV_HEARING_BASE varchar,
	PREV_IJ_CODE varchar,
	TRANS_NBR varchar,
	HEARING_DATE varchar,
	HEARING_TIME varchar,
	DEC_TYPE varchar,
	DEC_CODE varchar,
	DEPORTED_1 varchar,
	DEPORTED_2 varchar,
	OTHER_COMP varchar,
	APPEAL_RSVD varchar,
	APPEAL_NOT_FILED varchar,
	COMP_DATE varchar,
	ABSENTIA varchar,
	VENUE_CHG_GRANTED varchar,
	TRANSFER_TO varchar,
	DATE_APPEAL_DUE_STATUS varchar,
	TRANSFER_STATUS varchar,
	CUSTODY varchar,
	CASE_TYPE varchar,
	NAT varchar,
	LANG varchar,
	SCHEDULED_HEAR_LOC varchar,
	CORRECTIONAL_FAC varchar,
	CR

In [4]:
filepath = os.path.join(folder, 'tbl_RepsAssigned.csv')

with open(filepath, 'rb') as f:
    first_line = f.readline().decode('utf-8')

columns = first_line.strip().split('\t')
print(columns)

['IDNREPSASSIGNED', 'IDNCASE', 'STRATTYLEVEL', 'STRATTYTYPE', 'PARENT_TABLE', 'PARENT_IDN', 'BASE_CITY_CODE', 'INS_TA_DATE_ASSIGNED', 'E_27_DATE', 'E_28_DATE', 'BLNPRIMEATTY']


In [9]:
f2 = r'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup'

In [14]:
lookup_files = ['tblLookupNationality.csv', 'tblLookupBaseCity.csv', 'tblLookupJudge.csv', 'tblLanguage.csv',
                'tblLookupCaseType.csv', 'tblLookupCustodyStatus.csv', 'tblLookupCourtDecision.csv', 'tbllookupAppealType.csv',
                'tblLookupBIADecision.csv', 'tblLookup_CasePriority.csv', 'tbllookupCharges.csv', 'tblLookUp_Appln.csv',
                'tblLookupSex.csv', 'tblLookupFiledBy.csv']

for filename in lookup_files:
    filepath = os.path.join(f2, filename)

    with open(filepath, 'rb') as f:
        first_line = f.readline().decode('utf-8')

    columns = first_line.strip().split('\t')

    print(f'\n{filename}: {len(columns)}')

    for i, col in enumerate(columns):
        print(f'\t{col} varchar,')


tblLookupNationality.csv: 8
	ID varchar,
	NAT_CODE varchar,
	NAT_NAME varchar,
	NAT_COUNTRY_NAME varchar,
	NAT_NACARA_COUNTRY varchar,
	datCreatedOn varchar,
	datModifiedOn varchar,
	blnActive varchar,

tblLookupBaseCity.csv: 98
	idnBaseCity varchar,
	BASE_CITY_CODE varchar,
	BASE_CITY_NAME varchar,
	BaseAutomated varchar,
	BASE_ST_ADDRESS varchar,
	BASE_CITY varchar,
	BASE_STATE varchar,
	BASE_ZIP_1 varchar,
	BASE_ZIP_2 varchar,
	BASE_PHONE_NO varchar,
	strRegion varchar,
	BaseEoir1Flag varchar,
	BaseEoir1Code varchar,
	BaseEoir1Suba varchar,
	BaseEoir1Subb varchar,
	BaseEoir1Subc varchar,
	BaseEoir1Subd varchar,
	BaseEoir1Sube varchar,
	BaseEoir1Subf varchar,
	BaseEoir2Flag varchar,
	BaseEoir2Code varchar,
	BaseEoir2Suba varchar,
	BaseEoir2Subb varchar,
	BaseEoir2Subc varchar,
	BaseEoir2Subd varchar,
	BaseEoir2Sube varchar,
	BaseEoir2Subf varchar,
	BaseEoir5Flag varchar,
	BaseEoir5Code varchar,
	BaseEoir5Suba varchar,
	BaseEoir5Subb varchar,
	BaseEoir5Subc varchar,
	BaseEoir5Subd va

In [25]:
lookup_files = ['tblLookupNationality.csv', 'tblLookupBaseCity.csv', 'tblLookupJudge.csv', 'tblLanguage.csv',
                'tblLookupCaseType.csv', 'tblLookupCustodyStatus.csv', 'tblLookupCourtDecision.csv', 'tbllookupAppealType.csv',
                'tblLookupBIADecision.csv', 'tblLookup_CasePriority.csv', 'tbllookupCharges.csv', 'tblLookUp_Appln.csv',
                'tblLookupSex.csv', 'tblLookupFiledBy.csv']

for filename in lookup_files:
    print(f"from '{str(f2)}\{filename}'")

from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLookupNationality.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLookupBaseCity.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLookupJudge.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLanguage.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLookupCaseType.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLookupCustodyStatus.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLookupCourtDecision.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tbllookupAppealType.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLookupBIADecision.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLookup_CasePriority.csv'
from 'C:\Users\dkim2\Desktop\projects\EOIR Case Data 202

In [ ]:
files = ['tblLookUp_Appln.csv', 'tblLookupCourtDecision.csv']

[]


In [28]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [30]:
import pandas as pd

In [32]:
df = pd.read_csv(r'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tblLookUp_Appln.csv', delimiter='\t')

In [37]:
df[3:6]

,idnAppln,strcode,strdescription,strAllowedCaseTypes,strAllowedDecisionCodes,APL_casetype_1,APL_casetype_2,APL_casetype_3,APL_casetype_4,APL_casetype_5,APL_casetype_6,APL_casetype_7,datCreatedOn,datModifiedOn,blnActive
3,4,212a,WAIVER OF UNLAWFUL PRESENCE UNDER 212(a)(9)(B)(v),"DEP,WHO","A,D,G,O,W,S,T,M,I,P",D,NaN,NaN,NaN,NaN,NaN,NaN,2003-08-07 00:00:00.000,NaN,1
4,5,212c,212C; WAIVER IN RMV PROCEED. PERMITTED BY CASE...,"RMV, EXC, DEP","A,D,G,O,W,S,T,M,I,P",M,E,D,NaN,NaN,NaN,NaN,2003-08-07 00:00:00.000,2009-10-22 11:26:36.000,1
5,6,212d,REQUEST FOR A WAIVER BY NONIMMIGRANT,"EXC,RMV","A,D,G,O,W,S,T,M,I,P",E,M,NaN,NaN,NaN,NaN,NaN,2003-08-07 00:00:00.000,NaN,1


In [38]:
files = ['tblLookUp_Appln.csv', 'tblLookupCourtDecision.csv']
cleaned_list = []

for filename in os.listdir(f2):

    if filename in files:
        filepath = os.path.join(f2, filename)
        final_path = os.path.join(f2, filename.replace('.csv', '_final.csv'))

        print(f'Currently processing: {filename}')

        # strip null bytes
        with open(filepath, 'rb') as f:
            data = f.read().replace(b'\x00', b'')

        lines = data.decode('utf-8', errors='replace').splitlines()

        # detect column count from header
        header = lines[0].strip().split('\t')
        num_cols = len(header)
        print(f'  Columns: {num_cols}')

        # normalize all rows
        with open(final_path, 'w', newline='', encoding='utf-8') as outfile:
            writer = csv.writer(outfile, delimiter='\t')

            for line in lines:
                cols = line.strip().split('\t')

                if len(cols) > num_cols:
                    cols = cols[:num_cols]

                elif len(cols) < num_cols:
                    cols += [''] * (num_cols - len(cols))
                    
                writer.writerow(cols)

        cleaned_file = filename.replace(".csv", "_final.csv")
        cleaned_list.append(cleaned_file)

        print(f'{filename.replace(".csv", "_final.csv")} --- complete.')

print(cleaned_list)

Currently processing: tblLookupCourtDecision.csv
  Columns: 9
tblLookupCourtDecision_final.csv --- complete.
Currently processing: tblLookUp_Appln.csv
  Columns: 15
tblLookUp_Appln_final.csv --- complete.
['tblLookupCourtDecision_final.csv', 'tblLookUp_Appln_final.csv']


In [39]:
appeal_type_df = pd.read_csv(r'C:\Users\dkim2\Desktop\projects\EOIR Case Data 2026-0401\Lookup\tbllookupAppealType.csv', delimiter='\t')

In [40]:
appeal_type_df.columns

Index(['idnAppealType', 'strApplCode', 'strApplDescription', 'datCreatedOn',
       'datModifiedOn', 'blnActive', 'blnDDAppeal'],
      dtype='object')